# Démonstration de l'API — Plateforme d'analyse d'images médicales

Ce notebook illustre l'utilisation de l'API REST pour la chaîne complète de traitement :
1. Upload d'une image
2. Prétraitement
3. Segmentation classique
4. Segmentation deep learning (U-Net)
5. MC Dropout (quantification d'incertitude)
6. Évaluation des résultats

**Prérequis** : le backend doit tourner sur `http://localhost:8000`.

In [ ]:
import requests
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO

BASE = "http://localhost:8000/api"

def show_image(image_id, title=""):
    """Affiche une image depuis l'API."""
    r = requests.get(f"{BASE}/image/{image_id}")
    img = Image.open(BytesIO(r.content))
    plt.figure(figsize=(4, 4))
    plt.imshow(img, cmap="gray")
    plt.title(title, fontsize=10)
    plt.axis("off")
    plt.show()

## 1. Upload d'une image

In [ ]:
# Remplacez par le chemin de votre image
IMAGE_PATH = "../path/to/your/image.png"

with open(IMAGE_PATH, "rb") as f:
    resp = requests.post(f"{BASE}/upload", files={"file": f})

upload = resp.json()
image_id = upload["image_id"]
print(f"Image ID: {image_id}, Dimensions: {upload['width']}x{upload['height']}")
show_image(image_id, "Image originale")

## 2. Prétraitement

In [ ]:
# Appliquer un filtre CLAHE
resp = requests.post(f"{BASE}/preprocess/clahe", json={
    "image_id": image_id,
    "params": {"clip_limit": 3.0, "tile_grid_size": 8}
})
clahe_result = resp.json()
print(f"CLAHE appliqué en {clahe_result['duration_ms']} ms")
show_image(clahe_result["result_id"], "Après CLAHE")

In [ ]:
# Histogramme de l'image
resp = requests.get(f"{BASE}/preprocess/histogram/{image_id}")
hist_data = resp.json()["histograms"]

plt.figure(figsize=(6, 2))
for channel, values in hist_data.items():
    plt.plot(values, label=channel, linewidth=0.8)
plt.xlabel("Intensité", fontsize=9)
plt.ylabel("Fréquence", fontsize=9)
plt.legend(fontsize=8)
plt.title("Histogramme", fontsize=10)
plt.tight_layout()
plt.show()

## 3. Segmentation classique

In [ ]:
preprocessed_id = clahe_result["result_id"]

methods = ["otsu", "watershed", "chan_vese", "kmeans"]
results = {}

for method in methods:
    resp = requests.post(f"{BASE}/segment/classical/{method}", json={
        "image_id": preprocessed_id,
        "params": {}
    })
    results[method] = resp.json()
    print(f"{method}: {results[method]['duration_ms']} ms")

fig, axes = plt.subplots(1, len(methods), figsize=(3 * len(methods), 3))
for ax, method in zip(axes, methods):
    r = requests.get(f"{BASE}/image/{results[method]['mask_id']}")
    img = Image.open(BytesIO(r.content))
    ax.imshow(img, cmap="gray")
    ax.set_title(f"{method}\n{results[method]['duration_ms']}ms", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Segmentation deep learning (U-Net)

In [ ]:
resp = requests.post(f"{BASE}/segment/deep/unet", json={
    "image_id": preprocessed_id,
    "modality": "endoscopy"  # ou "brain"
})

if resp.status_code == 200:
    unet_result = resp.json()
    print(f"U-Net: {unet_result['duration_ms']} ms")
    show_image(unet_result["mask_id"], "Segmentation U-Net")
else:
    print(f"Erreur: {resp.json()['detail']}")
    print("Le modèle .hdf5 n'est probablement pas chargé.")

## 5. MC Dropout (Quantification d'incertitude)

In [ ]:
resp = requests.post(f"{BASE}/segment/deep/unet_mc", json={
    "image_id": preprocessed_id,
    "modality": "endoscopy",
    "n_passes": 30
})

if resp.status_code == 200:
    mc_result = resp.json()
    print(f"MC Dropout ({mc_result['n_passes']} passes): {mc_result['duration_ms']} ms")
    print(f"Confiance: {mc_result['confidence_score']}")

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, (key, title) in zip(axes, [
        ("mask_id", "Prédiction moyenne"),
        ("std_map_id", "Carte d'écart-type (σ)"),
        ("entropy_map_id", "Carte d'entropie"),
    ]):
        r = requests.get(f"{BASE}/image/{mc_result[key]}")
        img = Image.open(BytesIO(r.content))
        ax.imshow(img)
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    plt.suptitle(f"Confiance: {mc_result['confidence_score']:.2f}", fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print(f"Erreur: {resp.json()['detail']}")

## 6. Évaluation (si ground truth disponible)

In [ ]:
# Charger un masque ground truth
GT_PATH = "../path/to/your/mask.png"

with open(GT_PATH, "rb") as f:
    resp = requests.post(f"{BASE}/upload", files={"file": f})
gt_id = resp.json()["image_id"]

# Évaluer la segmentation U-Net
if 'unet_result' in dir():
    resp = requests.post(f"{BASE}/evaluate", json={
        "mask_pred_id": unet_result["mask_id"],
        "mask_truth_id": gt_id
    })
    metrics = resp.json()
    print("Métriques d'évaluation :")
    for k, v in metrics.items():
        print(f"  {k}: {v}")

## 7. Comparaison multi-méthodes

In [ ]:
resp = requests.post(f"{BASE}/compare", json={
    "image_id": preprocessed_id,
    "methods": ["otsu", "adaptive_threshold", "watershed", "chan_vese", "kmeans", "meanshift"],
    "ground_truth_id": gt_id if 'gt_id' in dir() else None
})

compare_results = resp.json()["results"]
for r in compare_results:
    line = f"{r['method']}: {r['duration_ms']}ms"
    if r.get('metrics'):
        line += f" — Dice={r['metrics']['dice']:.4f}, IoU={r['metrics']['iou']:.4f}"
    print(line)